# 05 - The photon transfer curve

**Purpose.** Run `protocols/02-ptc.md` and publish `g(gain)` - electrons per ADC count, per CFA
plane, at nine gain settings - together with the gain law fitted through it and the FPN test that
says whether sigma^2 is shot plus read and nothing else. It is the session that puts a scale on
every count session 01 measured.

**What it is not for.** Linearity, `ceiling(gain)` and full well: those need a characterised
light source, a shrunk ROI and a per-channel bend, and are a later session. Nor dark current.
Nothing here fits a bend, and no rung is placed to find one.

**This notebook captures; `06_ptc_read` explains.** The reading half publishes to `results/`.
Everything below either talks to the camera or decides what to ask it for next.

## 1. Pre-flight, and the attenuation scout

**This section is `light-source.md` item 3, and it runs *warm*, before and apart from the session
proper.** It answers the one question the protocol cannot answer on paper: *what grey level and
how many sheets* put the bench where a twelve-rung ladder fits between the camera's shortest
exposure and a sane wall clock.

It is deliberately not gate 3. Gate 3 solves `t_sat(gain)` from the measured flux *inside* the
session, cold, with the bench undisturbed, and that is what the record quotes. This scout produces
a **bench configuration** - a sheet count and a grey level - plus a prediction of `t_sat` that
gate 3 re-measures. Nothing here is published to `results/`.

**Why it may run warm.** Flux is an optical measurement and the sensor's temperature does not
change how much light arrives. What temperature does change - dark current - is held down by
keeping the scout's exposures short, and none of its numbers survive into the record.

### The arithmetic that sets the target

Gain is in 0.1 dB units, so amplification is `10 ** (gain/200)` and **gain 450 is 178x gain 0**.
(600 would have been 1000x, and is out of this project's scope - `CLAUDE.md`, D52. Dropping it is
what brings the attenuation this bench needs from ~4000x down to ~800x.) With one fixed light
level `t_sat` spans 178:1 across the gain set, and the ladder spans another 300:1 (0.3% to 90%),
so the session's shortest exposure is

    0.003 * t_sat(450)  =  1.69e-5 * t_sat(0)

so `t_sat(0)` alone fixes both the margin over the camera's 32 us floor and the wall clock, which
is about `25 * t_sat(0)` of shutter-open time.

| `t_sat(0)` | faintest rung | margin over 32 us | session exposure |
|---|---|---|---|
| 1.9 s | 32 us | none | 48 s |
| 5 s | 84 us | 2.6x | 2.1 min |
| 12 s | 202 us | 6.3x | 5.0 min |
| 30 s | 506 us | 16x | 12.6 min |

**Target: `t_sat(0)` between 10 and 30 s.**

In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, spatial, stats

RESULTS = pathlib.Path.cwd().parent / "results"

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
GAINS = [0, 50, 100, 190, 200, 250, 300, 450]   # 450 is the ceiling (D52)
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32, 54, 90]   # % of t_sat
ROI = (1408, 568, 1024, 1024)        # even origin and extent, or Bayer shifts (L05)
OFFSET = 15                          # project_offset, fixed by session 01
SCOUT_GAIN = 100
MAX_EXPOSURE = 2.0                   # s; the scout never needs a long frame

# The bench configuration this run measures.  A flux with no configuration
# beside it is not a measurement of anything (light-source.md item 3).
SHEETS = 8                           # sheets of paper between camera and panel
REFRESH_HZ = 60.0                    # panel refresh; one period is the flicker yardstick
PATCH_SERVER = "http://127.0.0.1:8765"

TARGET_TSAT0 = (10.0, 30.0)          # s, the window the table above argues for

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
PEDESTAL_FIT = _bias["pedestal_fit"]["value"]
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]


def amplification(gain):
    """Gain is in 0.1 dB units, so 200 units is exactly a factor of ten."""
    return 10.0 ** (gain / 200.0)


def pedestal(gain):
    """Published pedestal at offset 15, in ADC counts (03_bias_sweep)."""
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * amplification(gain)


def plane_means(mosaic):
    """Mean of each CFA plane, in ADC counts.  `to_adc` raises rather than
    truncate, so this doubles as a check that the frame came off the raw path."""
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def set_level(level, settle_s=0.8):
    """Drive `grey-patch.html` from here (protocols/patch-server.py).

    The page polls and applies the change, so the settle covers one poll plus a
    repaint.  Whether the panel actually followed is not taken on trust: the
    sweep below reads it back out of the pixels.
    """
    with urllib.request.urlopen(f"{PATCH_SERVER}/set?level={int(level)}", timeout=5) as r:
        state = json.load(r)
    time.sleep(settle_s)
    return state


print(f"bench: {SHEETS} sheets of paper, grey level driven from here")
print(f"min exposure {MIN_EXPOSURE * 1e6:.0f} us")
print("pedestal at offset 15:  " + "  ".join(f"g{g}={pedestal(g):.0f}" for g in GAINS))

### Gate 1 - white balance, verified from the pixels (L01)

The camera ships `WB_R=55`, `WB_B=75` and applies them to RAW16 before the data reaches us. The
control reading back as 50 proves only that the control took. The evidence is the modal step
between adjacent distinct values: **16 on all four planes**, greens at 16 with red 17/18 and blue
24 being the fingerprint of white balance still applied.

Nothing captured before this passes is usable - the scout included, because a smeared step means
`to_adc` refuses and every mean below is wrong.

In [ ]:
rig = asi.open_camera()
asi.neutralise_white_balance(rig)
asi.configure(rig, gain=SCOUT_GAIN, offset=OFFSET, roi=ROI)

dark, hdr = asi.capture(rig, MIN_EXPOSURE, imagetyp="DARK")
steps = {name: stats.value_step(p) for name, p in spatial.split(dark).items()}

print("modal value step per plane:", steps)
print("WB_R", rig.get("WB_R"), " WB_B", rig.get("WB_B"),
      " gain", rig.get("Gain"), " offset", rig.get("Offset"))
assert set(steps.values()) == {16}, f"gate 1 FAILED: {steps} -- stop, do not correct later"
print("\ngate 1 passed")

### Measuring a flux

One helper, used by everything below: step the exposure until the brightest plane lands near
mid-scale, then read flux off as `(mean - pedestal) / exptime`.

Mid-scale rather than near full scale on purpose. A plane close to 4095 is compressed by whatever
non-linearity lives near saturation, and this session is explicitly not the one that measures a
bend; half scale keeps the estimate on the part of the curve we are entitled to call straight.
One frame is discarded after every exposure change, as the protocol requires.

In [ ]:
ped_scout = pedestal(SCOUT_GAIN)


def auto_expose(start=1e-3, tries=9):
    """Land the brightest plane between 35% and 65% of full scale.

    Returns `(means, exptime, converged)`.  A run that does not converge is
    still returned rather than raised on: at the dim end the honest outcome is
    "as long as this scout will go and still not bright", and that is data.
    """
    exposure, means, exptime = start, None, start
    for _ in range(tries):
        asi.capture(rig, exposure, imagetyp="LIGHT")          # discard after the change
        mosaic, h = asi.capture(rig, exposure, imagetyp="LIGHT")
        means, exptime = plane_means(mosaic), h["EXPTIME"]
        top = max(means.values())
        if 0.35 * FULL_SCALE <= top <= 0.65 * FULL_SCALE:
            return means, exptime, True
        scale = 0.5 * FULL_SCALE / max(top - ped_scout, 1.0)
        nxt = min(max(exposure * scale, MIN_EXPOSURE), MAX_EXPOSURE)
        if abs(nxt - exposure) / exposure < 0.02:             # pinned at a limit
            break
        exposure = nxt
    return means, exptime, False


def flux_of(means, exptime):
    return {k: (v - ped_scout) / exptime for k, v in means.items()}


set_level(255)
means, exptime, ok = auto_expose()
flux = flux_of(means, exptime)
bright = max(flux, key=flux.get)

print(f"anchor: grey level 255, {SHEETS} sheets, gain {SCOUT_GAIN}, "
      f"{exptime * 1e6:.0f} us, converged={ok}")
for k, v in flux.items():
    print(f"  {k}: {v:12.1f} counts/s   ({v / flux[bright] * 100:5.1f}% of {bright})")

### What that flux implies

`t_sat(gain)` is where the **brightest** plane fills the headroom above its own pedestal -
brightest, because that is the plane that clips first and clipping is what the top rung must
avoid. The pedestal is not a detail here: at gain 600 it is over 1000 counts, a quarter of full
scale. At gain 450, the top of this project's range, it is 231 counts and the headroom is 3864.

In [ ]:
def tsat_table(flux_ref, label=""):
    amp_ref = amplification(SCOUT_GAIN)
    print(f"{label}\n{'gain':>5} {'amp':>8} {'pedestal':>9} {'headroom':>9} "
          f"{'t_sat':>12} {'0.3% rung':>12} {'90% rung':>12}")
    out = {}
    for g in GAINS:
        head = FULL_SCALE - pedestal(g)
        t = head / (flux_ref * amplification(g) / amp_ref)
        out[g] = t
        print(f"{g:5d} {amplification(g):8.1f} {pedestal(g):9.1f} {head:9.1f} "
              f"{t:11.4g}s {RUNGS[0] / 100 * t * 1e6:11.4g}u {RUNGS[-1] / 100 * t:11.4g}s")

    t0, ttop = out[0], out[GAINS[-1]]
    shortest = RUNGS[0] / 100 * ttop
    print(f"\nt_sat(gain 0)        {t0:12.4g} s    (target "
          f"{TARGET_TSAT0[0]:.0f}-{TARGET_TSAT0[1]:.0f} s)")
    print(f"shortest rung        {shortest * 1e6:12.4g} us   (floor {MIN_EXPOSURE * 1e6:.0f} us)")
    print(f"session shutter-open {25.1 * t0 / 60:12.4g} min")
    want = sum(TARGET_TSAT0) / 2
    print(f"attenuation to reach t_sat(0) = {want:.0f} s: {want / t0:.4g}x")
    return out


tsat_table(flux[bright], f"bench as it stands (level 255, {SHEETS} sheets; "
                         f"{bright} sets t_sat):")

### The grey-level curve, measured rather than assumed

L07 says grey level is exhausted below about 25% of full scale, because the backlight leaks
through a black LCD. That is a claim from a retired attempt about a different bench, and it is
cheap to check here: the level is driven from this notebook, so the curve costs no trips to the
iPad.

Two things come out of it. The **usable range** - how much attenuation the level alone can
deliver before the leak floor - and a check that the panel is actually following: if the page is
not connected to the server, every level returns the same flux and the table below is flat.

In [ ]:
LEVELS = [255, 224, 192, 160, 128, 96, 64, 48, 32, 24, 16, 8, 0]

curve = []
for lv in LEVELS:
    set_level(lv)
    m, e, converged = auto_expose(start=max(exptime, MIN_EXPOSURE))
    curve.append((lv, flux_of(m, e)[bright], e, converged))

f255 = curve[0][1]
print(f"{'level':>6} {'% of white':>11} {'flux':>14} {'attenuation':>12} "
      f"{'exptime':>10} {'converged':>10}")
for lv, f, e, converged in curve:
    print(f"{lv:6d} {lv / 255 * 100:10.1f}% {f:14.1f} {f255 / max(f, 1e-9):11.4g}x "
          f"{e * 1e6:9.0f}u {str(converged):>10}")

best = curve[-1][1]
print(f"\ngrey level alone gives {f255 / max(best, 1e-9):.4g}x, from 255 down to 0.")
assert curve[-1][1] < 0.5 * f255, ("level 0 is as bright as level 255 -- the panel is not "
                                   "following the server.  Reload grey-patch.html on the iPad.")

### Does the panel flicker?

If the backlight modulates, an exposure shorter than a refresh period samples a varying slice of
the cycle, the frame level jitters, and that jitter lands in the pair-difference variance where
it is indistinguishable from read noise. It would bias exactly the low rungs the geometric ladder
exists to constrain (L10).

The test needs no knowledge of `g`. Averaged over a quarter-million pixels shot noise moves a
plane mean by parts in ten thousand, so an excess that **shrinks as the exposure lengthens** is
the panel or the shutter and not the sensor. Probe exposures are fractions of the measured
`t_sat`, so nothing in this test can clip.

In [ ]:
VENDOR_G0 = 9.4      # e-/ADU at gain 0, read off ZWO's chart -- a prediction, not a constant


def burst(exposure_s, n=10):
    asi.capture(rig, exposure_s, imagetyp="LIGHT")            # discard after the change
    out = []
    for _ in range(n):
        m, h = asi.capture(rig, exposure_s, imagetyp="LIGHT")
        out.append(plane_means(m)[bright])
    return np.array(out)


set_level(255)
means, exptime, ok = auto_expose()
flux = flux_of(means, exptime)
t_sat_ref = (FULL_SCALE - ped_scout) / flux[bright]
period = 1.0 / REFRESH_HZ
g_here = VENDOR_G0 / amplification(SCOUT_GAIN)
npix = (ROI[3] // 2) * (ROI[2] // 2)

print(f"t_sat at gain {SCOUT_GAIN} is {t_sat_ref * 1e3:.4g} ms; "
      f"one refresh period is {period * 1e3:.1f} ms")
print(f"\n{'exposure':>12} {'periods':>9} {'signal':>10} {'scatter':>9} {'rel':>8} {'shot pred':>10}")
for frac in (0.01, 0.05, 0.20, 0.60):
    e = frac * t_sat_ref
    if e < MIN_EXPOSURE:
        print(f"{e * 1e6:10.1f} us   below the {MIN_EXPOSURE * 1e6:.0f} us floor -- skipped")
        continue
    b = burst(e)
    signal = b.mean() - ped_scout
    shot = np.sqrt(max(signal, 0.0) / g_here / npix) / b.mean() * 100
    print(f"{e * 1e6:10.0f} us {e / period:9.2f} {signal:10.1f} {b.std(ddof=1):9.3f} "
          f"{b.std(ddof=1) / b.mean() * 100:7.3f}% {shot:9.3f}%")

print(f"\nLast column: what shot noise alone would do to a plane mean of {npix:,} pixels,")
print("using ZWO's chart value for g as a prediction (a hypothesis, never a constant).")

### Record for the bench

The scout ends with a **configuration, not a constant**: grey level, sheet count, and the flux
that pair produced, written into the session record with the ambient temperature. The attenuation
is valid only while nobody moves the camera off the panel.

If the attenuation the bench can reach falls short of what `t_sat(0)` needs, the honest response
is not to fudge the ladder. It is to say which gains the source can support and shorten the gain
set at the top - and to record that gain 600 was dropped for want of light rather than quietly
shooting it with a ladder whose bottom rungs sit under the shutter floor.

In [ ]:
set_level(255)
rig.close()      # drops the cooler too, but the scout never turned it on
print("camera closed")